## Importación

In [27]:
from pathlib import Path
import sys
import yaml

import pandas as pd
import numpy as np
import pandas as pd
import random
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


BASE_DIR = Path().resolve().parent
sys.path.append(str(BASE_DIR/'src'))

## Rutas

In [28]:
DATA_FILE = BASE_DIR / 'data' / 'Feature_Selection' / '01_train.csv' 
CONFIG_FILE = BASE_DIR / 'config' / '01_experimento.yaml'

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_FILE: {DATA_FILE}")
print(f"CONFIG_FILE: {CONFIG_FILE}")

BASE_DIR: /home/jair/Proyectos/Loan_Status_Prediction
DATA_FILE: /home/jair/Proyectos/Loan_Status_Prediction/data/Feature_Selection/01_train.csv
CONFIG_FILE: /home/jair/Proyectos/Loan_Status_Prediction/config/01_experimento.yaml


## Configuraciónde hiperparpametros

In [29]:
with open(CONFIG_FILE, 'r') as f:
    config = yaml.safe_load(f)

seed = config['seed']
random.seed(seed)
np.random.seed(seed)

test_size = config['data_split']['test_size']
random_state = config['seed']
variable_objetivo = config['target_variable']

## Obtención datos

In [30]:
train = pd.read_csv(DATA_FILE)
train.head()

,age,education_level,annual_income,monthly_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,installment,grade_subgrade,...,loan_purpose_Debt consolidation,loan_purpose_Education,loan_purpose_Home,loan_purpose_Medical,loan_purpose_Other,loan_purpose_Vacation,loan_to_income,has_delinquency_history,sevetity_score,payment_income
0,28,2.0,10.751751,8.267079,0.271553,773,29588.02,9.79,951.81,1.0,...,0,1,0,0,0,0,29577.268249,1,1,115.132561
1,34,1.0,10.618642,8.134004,0.263133,775,500.00,9.69,16.06,1.0,...,0,0,1,0,0,0,489.381358,1,2,1.974427
2,60,1.0,9.514391,7.030291,0.175633,748,21230.36,10.29,687.94,1.0,...,0,0,0,0,1,0,21220.845609,1,1,97.853696
3,57,1.0,10.081107,7.596658,0.235072,648,6602.02,14.95,228.70,3.0,...,0,0,0,1,0,0,6591.938893,1,2,30.105342
4,23,1.0,10.966093,8.481375,0.079735,594,17108.03,14.48,588.71,4.0,...,0,0,0,0,1,0,17097.063907,1,1,69.412092


In [31]:
print(train.columns)

Index(['age', 'education_level', 'annual_income', 'monthly_income',
       'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate',
       'installment', 'grade_subgrade', 'num_of_open_accounts',
       'total_credit_limit', 'current_balance', 'delinquency_history',
       'public_records', 'num_of_delinquencies', 'loan_paid_back', 'age_group',
       'gender_Female', 'gender_Male', 'gender_Other', 'loan_term_60',
       'marital_status_Divorced', 'marital_status_Married',
       'marital_status_Single', 'marital_status_Widowed',
       'employment_status_Employed', 'employment_status_Retired',
       'employment_status_Self-employed', 'employment_status_Student',
       'employment_status_Unemployed', 'loan_purpose_Business',
       'loan_purpose_Car', 'loan_purpose_Debt consolidation',
       'loan_purpose_Education', 'loan_purpose_Home', 'loan_purpose_Medical',
       'loan_purpose_Other', 'loan_purpose_Vacation', 'loan_to_income',
       'has_delinquency_history', 's

In [32]:
print(len(train.columns.to_list()))

43


## Algoritmos

* modelo
* filtros
* wrapper

### Modelo

In [ ]:
X_train = train.drop(variable_objetivo, axis = 1)
y_train = train[variable_objetivo]

print(f'Dimensiones X: {X_train.shape}')
print(f'Dimensiones y: {y_train.shape}')

#### Regresion lineal - Solo captura relaciónes lineales

In [34]:
modelo = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter= config['linear_regression']['max_iter']))
modelo.fit(X_train, y_train)
importances = np.abs(modelo.named_steps['logisticregression'].coef_[0])
importances

array([1.87359952e-02, 1.67740879e-02, 1.99467361e-02, 2.03411005e-02,
       8.67261890e-01, 1.18357969e+00, 5.86502120e-03, 1.58075072e-02,
       5.72034051e-04, 3.07312439e-01, 2.21730586e-03, 9.60157350e-04,
       2.29348060e-02, 8.95567726e-02, 1.32453876e-02, 4.85777141e-02,
       2.81193306e-03, 1.03866999e-02, 1.61517053e-02, 1.99116484e-02,
       1.73620846e-03, 1.77839317e-02, 1.59566115e-02, 1.87944164e-02,
       3.59147544e-02, 3.19700538e-01, 9.21992290e-01, 2.31597396e-01,
       4.75851321e-01, 1.17082319e+00, 5.79872873e-03, 9.20827980e-03,
       1.73211359e-03, 6.21483968e-02, 6.15005822e-02, 2.87486432e-02,
       2.09366482e-02, 4.04674661e-03, 5.86362287e-03, 4.33043156e-03,
       4.55767030e-02, 2.58313858e-03])

In [ ]:
modelo.fit(X_train, y_train)
coeficientes = modelo.named_steps['logisticregression'].coef_[0]
importancias = np.abs(coeficientes)

nombres_caracteristicas = X_train.columns
tabla_importancias = pd.DataFrame({
    'Característica': nombres_caracteristicas,
    'Coeficiente': coeficientes,
    'Importancia': importancias
})

tabla_importancias = tabla_importancias.sort_values('Importancia', ascending=False)
tabla_importancias = tabla_importancias.reset_index(drop=True)
print(tabla_importancias)

                     Característica  Coeficiente  Importancia
0                      credit_score     1.183580     1.183580
1      employment_status_Unemployed    -1.170823     1.170823
2         employment_status_Retired     0.921992     0.921992
3              debt_to_income_ratio    -0.867262     0.867262
4         employment_status_Student    -0.475851     0.475851
5        employment_status_Employed     0.319701     0.319701
6                    grade_subgrade     0.307312     0.307312
7   employment_status_Self-employed     0.231597     0.231597
8               delinquency_history    -0.089557     0.089557
9            loan_purpose_Education    -0.062148     0.062148
10                loan_purpose_Home     0.061501     0.061501
11             num_of_delinquencies     0.048578     0.048578
12                   sevetity_score     0.045577     0.045577
13           marital_status_Widowed    -0.035915     0.035915
14             loan_purpose_Medical    -0.028749     0.028749
15      

#### Random Forest

In [ ]:
rfc = RandomForestClassifier(
    n_estimators= config['random_forest']['n_estimators'],
    max_leaf_nodes= config['random_forest']['n_estimators'],
    n_jobs= config['random_forest']['n_jobs']
)
scores = {}
rfc.fit(X_train, y_train)
for name, score in zip(train.columns, rfc.feature_importances_):
    scores[score] = name

desc = {k: v for k, v in sorted(scores.items(), key=lambda item: item[0], reverse=True)}
print('Score \t Característica')
print('='*45)
for key, value in desc.items():
    print(f'{np.round(key, 4)} \t {value}')

Score 	 Característica
0.2736 	 employment_status_Student
0.093 	 debt_to_income_ratio
0.0765 	 marital_status_Widowed
0.0654 	 credit_score
0.0511 	 employment_status_Self-employed
0.0424 	 employment_status_Retired
0.0328 	 interest_rate
0.0322 	 grade_subgrade
0.0318 	 employment_status_Employed
0.0247 	 current_balance
0.024 	 total_credit_limit
0.0232 	 annual_income
0.0231 	 monthly_income
0.0225 	 installment
0.0222 	 sevetity_score
0.0215 	 loan_purpose_Vacation
0.021 	 loan_amount
0.0202 	 age
0.0141 	 num_of_open_accounts
0.0103 	 delinquency_history
0.0094 	 has_delinquency_history
0.0093 	 num_of_delinquencies
0.0082 	 education_level
0.0066 	 loan_paid_back
0.0033 	 loan_purpose_Car
0.0031 	 gender_Female
0.0031 	 marital_status_Divorced
0.003 	 marital_status_Married
0.003 	 age_group
0.0027 	 gender_Other
0.0026 	 loan_purpose_Business
0.0024 	 loan_purpose_Medical
0.0024 	 loan_purpose_Debt consolidation
0.0021 	 loan_purpose_Education
0.0021 	 employment_status_Unemplo

## SHAP

#### SHAP VALUES

In [ ]:
#explainer = shap.Explainer(rfc)
#shap_values = explainer(X_train)
#np.shape(shap_values.values)

#### WATER FALL PLOT

In [ ]:
#print(type(shap_values))
#if isinstance(shap_values, list):
#    print(f"Número de clases: {len(shap_values)}")
#    print(f"Forma de cada array: {[v.shape for v in shap_values]}")

#### ABSOLUTE MEAN SHAP


In [ ]:
#shap_values_class1 = shap_values[:, :, 1]
#shap.plots.bar(shap_values_class1)